# LieWaves EEG — Grad-CAM Explainability

This notebook extends the original LieWaves CNN pipeline with **Grad-CAM** (Gradient-weighted Class Activation Mapping).

**What this notebook does:**
1. Loads and preprocesses data (same pipeline as original)
2. Trains a fixed CNN on DWT features
3. Applies Grad-CAM to identify **which EEG channels and frequency bands** drive the model's predictions
4. Visualises the results as heatmaps for neurophysiological interpretation

**Expected folder structure (relative to this notebook):**
```
Master_Thesis/
├── liewaves_gradcam.ipynb       ← this file
├── Truth_Sessions/
│   └── 1_BandPass_Filtered/
│       ├── subject1_truth.csv
│       ├── subject2_truth.csv
│       └── ...
└── Lie_Sessions/
    └── 1_BandPass_Filtered/
        ├── subject1_lie.csv
        ├── subject2_lie.csv
        └── ...
```

**Channels (Emotiv Insight):** AF3, AF4, T7, T8, Pz  
**Sampling rate:** 128 Hz

---
## Step 0: Install Dependencies

In [ ]:
# Install required packages if not already present
# Run once, then restart kernel if needed
%pip install scikit-learn PyWavelets tensorflow matplotlib seaborn

---
## Step 1: Imports & Global Configuration

All paths and hyperparameters are defined here in one place.
Changing these values will automatically propagate through the entire notebook.

In [ ]:
import os
import numpy as np
import pandas as pd
import pywt
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# ─────────────────────────────────────────────────────────
# DATA PATHS — adjust if your notebook is in a different folder
# These paths are relative to where you run the notebook from.
# Expected: Master_Thesis/ is the working directory.
# ─────────────────────────────────────────────────────────
TRUTH_PATH = 'Truth_Sessions/1_BandPass_Filtered/'
LIE_PATH   = 'Lie_Sessions/1_BandPass_Filtered/'

# ─────────────────────────────────────────────────────────
# EEG CHANNEL NAMES (Emotiv Insight, 5 channels)
# Used for labelling Grad-CAM heatmap axes
# ─────────────────────────────────────────────────────────
CHANNELS = ['AF3', 'AF4', 'T7', 'T8', 'Pz']

# ─────────────────────────────────────────────────────────
# SEGMENTATION PARAMETERS
# window_size = 128 samples = 1 second at 128 Hz
# overlap     = 64  samples = 50% overlap between windows
# ─────────────────────────────────────────────────────────
WINDOW_SIZE = 128
OVERLAP     = 64

# ─────────────────────────────────────────────────────────
# DWT PARAMETERS
# db4 (Daubechies 4) is standard for EEG — good balance of
# time/frequency resolution.
# level=4 gives sub-bands: cA4, cD4, cD3, cD2, cD1
# ─────────────────────────────────────────────────────────
WAVELET = 'db4'
LEVEL   = 4

# ─────────────────────────────────────────────────────────
# TRAINING PARAMETERS
# ─────────────────────────────────────────────────────────
EPOCHS     = 30
BATCH_SIZE = 32
TEST_SPLIT = 0.2   # 80% train, 20% test
RANDOM_SEED = 42

print("✓ Configuration loaded")
print(f"  Truth path : {TRUTH_PATH}")
print(f"  Lie path   : {LIE_PATH}")
print(f"  Channels   : {CHANNELS}")
print(f"  Wavelet    : {WAVELET}, Level {LEVEL}")
print(f"  Window     : {WINDOW_SIZE} samples, Overlap: {OVERLAP} samples")

---
## Step 2: Data Loading & Normalisation

Loads all CSV files from Truth and Lie session folders, then applies **z-score normalisation** (StandardScaler) so all channels have mean=0, std=1.  
This prevents channels with larger absolute voltages from dominating the model.

In [ ]:
def load_and_concatenate_data(path):
    """
    Reads all CSV files in `path` and concatenates them into one DataFrame.
    Each CSV is assumed to have columns for each EEG channel (no label column).
    """
    data_frames = []
    files = sorted([f for f in os.listdir(path) if f.endswith('.csv')])
    
    if len(files) == 0:
        raise FileNotFoundError(
            f"No CSV files found in: {path}\n"
            f"Current working directory: {os.getcwd()}\n"
            f"Make sure you run this notebook from the Master_Thesis/ folder."
        )
    
    for filename in files:
        df = pd.read_csv(os.path.join(path, filename))
        data_frames.append(df)
    
    print(f"  Loaded {len(files)} files from {path}")
    return pd.concat(data_frames, ignore_index=True)


# ── Load raw data ──────────────────────────────────────────
print("Loading data...")
truth_df = load_and_concatenate_data(TRUTH_PATH)
lie_df   = load_and_concatenate_data(LIE_PATH)

print(f"\nTruth data shape : {truth_df.shape}  (rows × channels)")
print(f"Lie data shape   : {lie_df.shape}")

# ── Normalise ──────────────────────────────────────────────
# NOTE: We fit the scaler on truth data only, then transform both.
# Alternatively, fit on combined — adjust if needed.
scaler = StandardScaler()
truth_data = scaler.fit_transform(truth_df)
lie_data   = scaler.transform(lie_df)   # use same scaler — important!

print("\n✓ Data loaded and normalised")

---
## Step 3: Segmentation (Sliding Window)

Splits the continuous EEG signal into overlapping windows.  
- **Window = 128 samples = 1 second** (at 128 Hz)  
- **Overlap = 64 samples = 50%** → each window shares half its data with the next  

This is standard in EEG ML pipelines and increases the number of training samples.

In [ ]:
def segment_data(data, window_size=WINDOW_SIZE, overlap=OVERLAP):
    """
    Splits a 2D array (time × channels) into overlapping windows.
    
    Args:
        data        : np.array of shape (n_samples, n_channels)
        window_size : number of time steps per window
        overlap     : step size between windows (window_size - overlap = stride)
    
    Returns:
        np.array of shape (n_windows, window_size, n_channels)
    """
    step = window_size - overlap   # stride = 64 → 50% overlap
    segments = []
    for start in range(0, len(data) - window_size, step):
        segment = data[start : start + window_size, :]
        segments.append(segment)
    return np.array(segments)


truth_segments = segment_data(truth_data)
lie_segments   = segment_data(lie_data)

print(f"Truth segments : {truth_segments.shape}  (n_windows × time × channels)")
print(f"Lie segments   : {lie_segments.shape}")

---
## Step 4: Feature Extraction with DWT

**Why DWT?**  
EEG signals contain meaningful information at different frequency bands (delta, theta, alpha, beta, gamma). DWT decomposes each channel into these bands simultaneously.

**What we extract:**  
For each channel, we run `pywt.wavedec` which returns 5 coefficient arrays:  
- `cA4` — approximation (lowest frequencies, ~0–4 Hz delta)
- `cD4` — detail level 4 (~4–8 Hz theta)
- `cD3` — detail level 3 (~8–16 Hz alpha)
- `cD2` — detail level 2 (~16–32 Hz beta)
- `cD1` — detail level 1 (~32–64 Hz gamma)

These are **flattened and concatenated** into a single feature vector per window.

> ⚠️ **Grad-CAM note:** We keep track of feature sizes per sub-band per channel,  
> so we can map Grad-CAM importance scores back to the original channel × sub-band space.

In [ ]:
def extract_dwt_features(segments, wavelet=WAVELET, level=LEVEL):
    """
    Extracts DWT features from each segment.
    
    For each segment:
      - For each channel: compute wavelet decomposition → flatten coefficients
      - Concatenate all channels → one feature vector per segment
    
    Args:
        segments : np.array of shape (n_windows, window_size, n_channels)
        wavelet  : wavelet family (e.g. 'db4')
        level    : decomposition depth
    
    Returns:
        features         : np.array of shape (n_windows, total_features)
        coeff_sizes      : list of int — number of coefficients per sub-band (same for all channels)
        sub_band_labels  : list of str — names like ['cA4','cD4','cD3','cD2','cD1']
    """
    features = []
    coeff_sizes = None
    sub_band_labels = None
    
    for seg_idx, segment in enumerate(segments):
        segment_features = []
        
        for ch in range(segment.shape[1]):
            # Decompose this channel's 1D signal
            coeffs = pywt.wavedec(segment[:, ch], wavelet, level=level)
            # coeffs is a list: [cA_level, cD_level, ..., cD1]
            
            # Save sub-band metadata on first segment/channel
            if seg_idx == 0 and ch == 0:
                coeff_sizes = [len(c) for c in coeffs]
                sub_band_labels = [f'cA{level}'] + [f'cD{l}' for l in range(level, 0, -1)]
            
            # Flatten all sub-bands for this channel into one array
            segment_features.append(np.hstack(coeffs))
        
        # Stack all channels into one long feature vector
        features.append(np.hstack(segment_features))
    
    return np.array(features), coeff_sizes, sub_band_labels


truth_features, coeff_sizes, sub_band_labels = extract_dwt_features(truth_segments)
lie_features, _, _ = extract_dwt_features(lie_segments)

n_channels   = truth_segments.shape[2]
features_per_channel = sum(coeff_sizes)

print(f"Sub-band labels : {sub_band_labels}")
print(f"Coefficients per sub-band : {coeff_sizes}")
print(f"Features per channel : {features_per_channel}")
print(f"Total feature vector length : {n_channels} channels × {features_per_channel} = {n_channels * features_per_channel}")
print(f"\nTruth feature matrix : {truth_features.shape}")
print(f"Lie feature matrix   : {lie_features.shape}")

---
## Step 5: Prepare Train/Test Split

Labels: **1 = Truth, 0 = Lie**  
We use a **stratified 80/20 split** to ensure both classes are equally represented in both sets.

In [ ]:
# ── Create labels ──────────────────────────────────────────
truth_labels = np.ones(truth_features.shape[0])   # 1 = Truth
lie_labels   = np.zeros(lie_features.shape[0])    # 0 = Lie

# ── Combine into X and y ───────────────────────────────────
X = np.vstack((truth_features, lie_features))
y = np.hstack((truth_labels, lie_labels))

# ── Train/test split (stratified) ─────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SPLIT,
    random_state=RANDOM_SEED,
    stratify=y
)

# ── Reshape for CNN: (samples, features, 1) ───────────────
# Conv1D expects 3D input: (batch, timesteps, channels)
# Here: timesteps = feature vector length, channels = 1
X_train_cnn = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test_cnn  = X_test.reshape(X_test.shape[0],  X_test.shape[1],  1)

print(f"Train set : {X_train_cnn.shape}  → {np.bincount(y_train.astype(int))} [Lie, Truth]")
print(f"Test set  : {X_test_cnn.shape}   → {np.bincount(y_test.astype(int))} [Lie, Truth]")

---
## Step 6: Build the CNN

The architecture follows the original notebook but with two important fixes:

| Issue in original | Fix here |
|---|---|
| `Dense(2, softmax)` + `Dense(1, sigmoid)` at the end — conflicting heads | Single `Dense(1, sigmoid)` output for binary classification |
| `callbacks` variable name conflicts with `from tensorflow.keras import callbacks` | Renamed callback list to `cb_list` |

**Architecture:** 3 × Conv1D → Flatten → Dense(256) → Dense(128) → Dense(64) → Dense(1)  
**Loss:** Binary cross-entropy  
**Output:** Sigmoid probability (0 = Lie, 1 = Truth)

In [ ]:
def build_cnn(input_length):
    """
    Builds a 1D CNN for binary EEG classification.
    
    The model has a named last conv layer ('last_conv') which we need
    for Grad-CAM — gradients will be computed with respect to this layer.
    
    Args:
        input_length : int — length of the DWT feature vector
    
    Returns:
        Compiled Keras model
    """
    model = models.Sequential([
        # ── Stage 1: 256 filters, kernel size 3 ──────────────
        layers.Conv1D(256, 3, activation='relu',
                      input_shape=(input_length, 1), name='conv1'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(2),
        layers.Dropout(0.25),
        
        # ── Stage 2: 128 filters ─────────────────────────────
        layers.Conv1D(128, 3, activation='relu', name='conv2'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(2),
        layers.Dropout(0.25),
        
        # ── Stage 3: 64 filters — THIS IS THE Grad-CAM target ─
        # We name it 'last_conv' so we can reference it easily later.
        layers.Conv1D(64, 3, activation='relu', name='last_conv'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(2),
        layers.Dropout(0.25),
        
        # ── Fully connected head ──────────────────────────────
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dense(128, activation='relu'),
        layers.Dense(64,  activation='relu'),
        
        # ── Output: single neuron, sigmoid = probability of Truth ─
        layers.Dense(1, activation='sigmoid', name='output')
    ])
    
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model


model = build_cnn(input_length=X_train_cnn.shape[1])
model.summary()

---
## Step 7: Train the Model

Uses **EarlyStopping** to stop if validation accuracy stops improving, and **ModelCheckpoint** to save the best weights.

⏱️ Training takes ~5–15 minutes on CPU depending on your dataset size.

In [ ]:
# ── Callbacks ──────────────────────────────────────────────
# NOTE: The variable is named 'cb_list' (not 'callbacks') to avoid
# shadowing the 'callbacks' module we imported from Keras above.
cb_list = [
    callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ModelCheckpoint(
        'best_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=0
    )
]

# ── Train ──────────────────────────────────────────────────
history = model.fit(
    X_train_cnn, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_test_cnn, y_test),
    callbacks=cb_list,
    verbose=1
)

print("\n✓ Training complete")

---
## Step 8: Evaluate the Model

Standard evaluation: accuracy, precision, recall, F1-score, and confusion matrix.

In [ ]:
# ── Make predictions ───────────────────────────────────────
y_pred_prob = model.predict(X_test_cnn, verbose=0)
y_pred      = (y_pred_prob > 0.5).astype(int).flatten()

# ── Classification report ──────────────────────────────────
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Lie', 'Truth'], zero_division=1))
print(f"F1 Score (macro): {f1_score(y_test, y_pred, average='macro', zero_division=1):.4f}")

# ── Confusion matrix ───────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred: Lie', 'Pred: Truth'],
            yticklabels=['True: Lie', 'True: Truth'], ax=ax)
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

# ── Training curves ────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['accuracy'],     label='Train')
ax1.plot(history.history['val_accuracy'], label='Validation')
ax1.set_title('Accuracy over Epochs')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()

ax2.plot(history.history['loss'],     label='Train')
ax2.plot(history.history['val_loss'], label='Validation')
ax2.set_title('Loss over Epochs')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

---
## Step 9: Grad-CAM Implementation

### How Grad-CAM works here:

1. **Target layer:** `last_conv` — the final Conv1D layer before the dense head.  
   This layer has already learned high-level EEG features and is directly before the decision.

2. **Forward pass with GradientTape:**  
   We record the activations of `last_conv` and the gradients of the output neuron w.r.t. those activations.

3. **Importance score per filter:**  
   `α_k = mean over time of (∂output / ∂feature_map_k)`  
   → filters that change the output the most get higher weight.

4. **Heatmap:**  
   `heatmap = ReLU( Σ_k α_k × feature_map_k )`  
   → Only positive contributions matter (features that push toward the predicted class).

5. **Map back to DWT features:**  
   The heatmap is over the `last_conv` output length. We resize it back to the original feature vector length, then split by channel and sub-band.

> **Why ReLU?** We only care about features that *increase* the prediction score.  
> Negative contributions would tell us about the opposite class, which is less informative here.

In [ ]:
# ── Build a model that outputs BOTH activations AND the final prediction ──
# This is the key trick: we need intermediate activations AND the final output
# in a single forward pass.

# Get the last conv layer by name (we named it 'last_conv' in build_cnn)
last_conv_layer = model.get_layer('last_conv')

# Build a sub-model that outputs [last_conv activations, final prediction]
grad_model = tf.keras.Model(
    inputs  = model.inputs,
    outputs = [last_conv_layer.output, model.output]
)

print(f"Grad-CAM model outputs:")
print(f"  last_conv output shape : {last_conv_layer.output.shape}  (None, time_steps, 64 filters)")
print(f"  final output shape     : {model.output.shape}            (None, 1)")


def compute_gradcam(sample, grad_model):
    """
    Computes the Grad-CAM heatmap for a single EEG feature vector.
    
    Args:
        sample     : np.array of shape (feature_length, 1)  — one window's features
        grad_model : Keras model outputting [conv_activations, prediction]
    
    Returns:
        heatmap : np.array of shape (feature_length,)
                  — importance score for each DWT feature position
        pred    : float — model prediction (0=Lie, 1=Truth)
    """
    # Add batch dimension: (1, feature_length, 1)
    sample_tensor = tf.cast(sample[np.newaxis, :, :], dtype=tf.float32)
    
    # ── GradientTape: record operations for gradient computation ──
    with tf.GradientTape() as tape:
        # We watch the conv layer outputs (not model weights!)
        tape.watch(sample_tensor)
        
        conv_outputs, predictions = grad_model(sample_tensor)
        # predictions shape: (1, 1)
        # conv_outputs shape: (1, T, 64)
        
        # We compute gradient w.r.t. the output score (before thresholding)
        class_score = predictions[:, 0]  # scalar
    
    # ── Compute gradients of output w.r.t. conv layer activations ──
    # grads shape: (1, T, 64)
    grads = tape.gradient(class_score, conv_outputs)
    
    # ── Pool gradients over the time dimension ──────────────────
    # α_k = mean over time of |∂output / ∂feature_map_k|
    # Result shape: (64,) — one weight per filter
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1))  # (64,)
    
    # ── Weighted combination of feature maps ────────────────────
    # conv_outputs[0] shape: (T, 64)
    # pooled_grads   shape: (64,)
    # Multiply each filter map by its importance weight
    conv_outputs_np = conv_outputs[0].numpy()  # (T, 64)
    pooled_grads_np = pooled_grads.numpy()     # (64,)
    
    for i in range(conv_outputs_np.shape[-1]):
        conv_outputs_np[:, i] *= pooled_grads_np[i]
    
    # ── Average over filters, apply ReLU ────────────────────────
    # ReLU keeps only features that *positively* contribute to the prediction
    heatmap = np.mean(conv_outputs_np, axis=-1)  # (T,)
    heatmap = np.maximum(heatmap, 0)              # ReLU
    
    # ── Normalise to [0, 1] ─────────────────────────────────────
    if heatmap.max() > 0:
        heatmap = heatmap / heatmap.max()
    
    # ── Resize heatmap to original feature vector length ────────
    # The conv+pooling layers reduce the time dimension.
    # We resize back to the original input length for interpretability.
    original_length = sample.shape[0]
    heatmap_resized = np.interp(
        np.linspace(0, len(heatmap) - 1, original_length),
        np.arange(len(heatmap)),
        heatmap
    )
    
    return heatmap_resized, float(predictions[0, 0])


print("\n✓ Grad-CAM functions defined")

---
## Step 10: Map Heatmap → Channel × Sub-band

The heatmap is currently one value per feature-vector position.  
Here we **split the heatmap** back into a 2D grid: **channels (rows) × sub-bands (columns)**.

This gives us a direct answer to:
> *"Which electrode contributes most to the model's Lie/Truth decision?  
> And at which frequency band?"*

In [ ]:
def heatmap_to_channel_subband(heatmap, n_channels, coeff_sizes):
    """
    Reshapes a flat heatmap into a (n_channels, n_sub_bands) importance matrix.
    
    The DWT feature vector is structured as:
        [CH1_cA4, CH1_cD4, CH1_cD3, CH1_cD2, CH1_cD1,
         CH2_cA4, CH2_cD4, ...,
         ...,
         CH5_cA4, ..., CH5_cD1]
    
    We average importance scores within each sub-band block per channel.
    
    Args:
        heatmap    : np.array of shape (total_features,)
        n_channels : int — number of EEG channels (5 for Emotiv Insight)
        coeff_sizes: list of int — number of coefficients per sub-band
    
    Returns:
        importance_matrix : np.array of shape (n_channels, n_sub_bands)
                            — mean importance per channel and sub-band
    """
    n_sub_bands = len(coeff_sizes)
    importance_matrix = np.zeros((n_channels, n_sub_bands))
    
    features_per_channel = sum(coeff_sizes)
    
    for ch_idx in range(n_channels):
        ch_start = ch_idx * features_per_channel
        
        band_start = ch_start
        for band_idx, band_size in enumerate(coeff_sizes):
            band_end = band_start + band_size
            # Average importance over all coefficients in this sub-band
            importance_matrix[ch_idx, band_idx] = np.mean(heatmap[band_start:band_end])
            band_start = band_end
    
    return importance_matrix


print("✓ Heatmap-to-channel-subband mapping function defined")
print(f"  Output grid will be {len(CHANNELS)} channels × {len(sub_band_labels)} sub-bands")

---
## Step 11: Run Grad-CAM on Test Samples

We run Grad-CAM on all **correctly classified** test samples and average the results.

Using only correctly classified samples is important because:  
- Misclassified samples may produce noisy or misleading gradients  
- We want to understand what the model has *correctly* learned

We compute separate heatmaps for **Lie** and **Truth** samples to compare which features are used for each class.

In [ ]:
print("Running Grad-CAM on test samples...")
print("(This may take a few minutes for large test sets)\n")

# Storage for importance matrices per class
lie_importances   = []   # correctly classified LIE samples
truth_importances = []   # correctly classified TRUTH samples

# ── Subsample for speed if test set is very large ──────────
# Change MAX_SAMPLES to None to run on all test samples
MAX_SAMPLES = 200  # set to None for full evaluation

indices = np.arange(len(X_test_cnn))
if MAX_SAMPLES is not None and len(indices) > MAX_SAMPLES:
    np.random.seed(RANDOM_SEED)
    indices = np.random.choice(indices, MAX_SAMPLES, replace=False)
    print(f"Subsampled to {MAX_SAMPLES} test samples for speed")

# ── Compute Grad-CAM for each sample ──────────────────────
for i, idx in enumerate(indices):
    sample     = X_test_cnn[idx]   # shape: (feature_length, 1)
    true_label = int(y_test[idx])  # 0=Lie, 1=Truth
    
    heatmap, pred = compute_gradcam(sample, grad_model)
    pred_label = 1 if pred > 0.5 else 0
    
    # Only use correctly classified samples
    if pred_label != true_label:
        continue
    
    # Map flat heatmap → (channel × sub-band) matrix
    importance_matrix = heatmap_to_channel_subband(heatmap, len(CHANNELS), coeff_sizes)
    
    if true_label == 0:
        lie_importances.append(importance_matrix)
    else:
        truth_importances.append(importance_matrix)
    
    if (i + 1) % 50 == 0:
        print(f"  Processed {i + 1}/{len(indices)} samples...")

# ── Average across samples ─────────────────────────────────
lie_avg   = np.mean(lie_importances,   axis=0) if lie_importances   else None
truth_avg = np.mean(truth_importances, axis=0) if truth_importances else None

print(f"\n✓ Grad-CAM complete")
print(f"  Lie samples used   : {len(lie_importances)}")
print(f"  Truth samples used : {len(truth_importances)}")

---
## Step 12: Visualise Grad-CAM Heatmaps

We plot three heatmaps:
1. **Lie class** — what the model focuses on when predicting "Lie"
2. **Truth class** — what the model focuses on when predicting "Truth"
3. **Difference** — which features are *selectively* used for Lie vs Truth

**How to interpret:**
- Bright/warm colours = high importance
- Channels to watch: **T7/T8** (temporal, associated with memory/cognitive load) and **Pz** (parietal, associated with P300 ERP)
- Sub-bands to watch: **cA4** (delta ~0–4Hz) and **cD4** (theta ~4–8Hz) — both linked to deception in literature

In [ ]:
def plot_gradcam_heatmap(importance_matrix, title, channel_names, sub_band_names, ax, vmin=None, vmax=None, cmap='YlOrRd'):
    """
    Plots a (n_channels, n_sub_bands) importance matrix as a heatmap.
    
    Args:
        importance_matrix : np.array of shape (n_channels, n_sub_bands)
        title             : string — plot title
        channel_names     : list of str — y-axis labels (EEG channels)
        sub_band_names    : list of str — x-axis labels (DWT sub-bands)
        ax                : matplotlib axis
        vmin, vmax        : colour scale limits (for consistent comparison across plots)
        cmap              : colormap
    """
    im = ax.imshow(importance_matrix, aspect='auto', cmap=cmap, vmin=vmin, vmax=vmax)
    plt.colorbar(im, ax=ax, label='Importance')
    
    ax.set_xticks(range(len(sub_band_names)))
    ax.set_xticklabels(sub_band_names, rotation=45, ha='right')
    ax.set_yticks(range(len(channel_names)))
    ax.set_yticklabels(channel_names)
    
    ax.set_xlabel('DWT Sub-band  (low freq → high freq)')
    ax.set_ylabel('EEG Channel')
    ax.set_title(title)
    
    # Add numerical values inside each cell
    for i in range(importance_matrix.shape[0]):
        for j in range(importance_matrix.shape[1]):
            ax.text(j, i, f'{importance_matrix[i, j]:.2f}',
                    ha='center', va='center', fontsize=8,
                    color='black' if importance_matrix[i, j] < 0.6 else 'white')


# ── Set consistent colour scale across all three plots ─────
all_values = []
if lie_avg   is not None: all_values.append(lie_avg)
if truth_avg is not None: all_values.append(truth_avg)
vmax = max(m.max() for m in all_values) if all_values else 1.0

# ── Plot ───────────────────────────────────────────────────
n_plots = 2 + (1 if lie_avg is not None and truth_avg is not None else 0)
fig, axes = plt.subplots(1, n_plots, figsize=(6 * n_plots, 5))
if n_plots == 1:
    axes = [axes]

plot_idx = 0

if lie_avg is not None:
    plot_gradcam_heatmap(lie_avg, 'Grad-CAM: LIE class\n(avg over correctly classified Lie samples)',
                         CHANNELS, sub_band_labels, axes[plot_idx], vmin=0, vmax=vmax)
    plot_idx += 1

if truth_avg is not None:
    plot_gradcam_heatmap(truth_avg, 'Grad-CAM: TRUTH class\n(avg over correctly classified Truth samples)',
                         CHANNELS, sub_band_labels, axes[plot_idx], vmin=0, vmax=vmax)
    plot_idx += 1

if lie_avg is not None and truth_avg is not None:
    diff = lie_avg - truth_avg
    plot_gradcam_heatmap(diff, 'Difference: LIE − TRUTH\n(positive = more important for Lie)',
                         CHANNELS, sub_band_labels, axes[plot_idx], 
                         vmin=-vmax, vmax=vmax, cmap='RdBu_r')

plt.suptitle('Grad-CAM Feature Importance: EEG Channel × DWT Sub-band', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('gradcam_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 13: Per-Channel Importance Bar Chart

A simpler view: **total importance per EEG channel** (summed over all sub-bands).  

This directly answers: *"Which electrode matters most for deception detection?"*

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

x = np.arange(len(CHANNELS))
width = 0.35

if lie_avg is not None:
    lie_per_channel = lie_avg.sum(axis=1)   # sum over sub-bands
    bars1 = ax.bar(x - width/2, lie_per_channel, width, label='Lie', color='#e74c3c', alpha=0.8)

if truth_avg is not None:
    truth_per_channel = truth_avg.sum(axis=1)
    bars2 = ax.bar(x + width/2, truth_per_channel, width, label='Truth', color='#2ecc71', alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(CHANNELS, fontsize=12)
ax.set_ylabel('Total Importance (summed over sub-bands)')
ax.set_title('Grad-CAM: Per-Channel Importance')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# ── Annotation: highlight neurophysiologically expected channels ──
ax.axvline(x=2, color='purple', linestyle='--', alpha=0.4, label='Temporal (T7,T8)')
ax.axvline(x=3, color='purple', linestyle='--', alpha=0.4)
ax.axvline(x=4, color='orange', linestyle=':', alpha=0.4, label='Parietal (Pz)')

# Add legend patches manually for the reference lines
ax.legend(handles=[
    mpatches.Patch(color='#e74c3c', alpha=0.8, label='Lie'),
    mpatches.Patch(color='#2ecc71', alpha=0.8, label='Truth'),
    plt.Line2D([0], [0], color='purple', linestyle='--', label='Temporal (T7, T8) — expected'),
    plt.Line2D([0], [0], color='orange', linestyle=':', label='Parietal (Pz) — P300 expected'),
])

plt.tight_layout()
plt.savefig('channel_importance.png', dpi=150)
plt.show()

---
## Step 14: Neurophysiological Interpretation

Automated check: does the model focus on scientifically expected regions?

In [ ]:
print("=" * 60)
print("NEUROPHYSIOLOGICAL VALIDATION SUMMARY")
print("=" * 60)

if lie_avg is not None and truth_avg is not None:
    avg_all = (lie_avg + truth_avg) / 2  # combined across both classes
    
    # ── Per-channel importance (normalised) ───────────────────
    channel_importance = avg_all.sum(axis=1)
    channel_importance_norm = channel_importance / channel_importance.sum()
    
    print("\n📊 Relative channel importance:")
    for ch, imp in zip(CHANNELS, channel_importance_norm):
        bar = '█' * int(imp * 40)
        print(f"  {ch:4s} : {bar:<40s} {imp:.1%}")
    
    # ── Per-sub-band importance (normalised) ──────────────────
    subband_importance = avg_all.sum(axis=0)
    subband_importance_norm = subband_importance / subband_importance.sum()
    
    print("\n📊 Relative sub-band importance:")
    freq_labels = ['delta (~0-4Hz)', 'theta (~4-8Hz)', 'alpha (~8-16Hz)', 
                   'beta (~16-32Hz)', 'gamma (~32-64Hz)']
    for sb, imp, freq in zip(sub_band_labels, subband_importance_norm, freq_labels):
        bar = '█' * int(imp * 40)
        print(f"  {sb:4s} [{freq:20s}] : {bar:<40s} {imp:.1%}")
    
    # ── Automated checks ──────────────────────────────────────
    print("\n📝 Automated checks (thesis RQ2):")
    
    # Check 1: Are temporal channels (T7/T8) important?
    t7_t8_idx = [CHANNELS.index('T7'), CHANNELS.index('T8')]
    temporal_imp = channel_importance_norm[t7_t8_idx].mean()
    print(f"  T7/T8 avg importance: {temporal_imp:.1%}  "
          f"{'✅ HIGH — consistent with temporal lobe deception studies' if temporal_imp > 0.15 else '⚠️  LOW — model may not use temporal features'}")
    
    # Check 2: Is Pz (P300 channel) important?
    pz_idx = CHANNELS.index('Pz')
    pz_imp = channel_importance_norm[pz_idx]
    print(f"  Pz importance:        {pz_imp:.1%}  "
          f"{'✅ HIGH — consistent with P300 ERP deception marker' if pz_imp > 0.15 else '⚠️  LOW — P300 not a primary driver'}")
    
    # Check 3: Are low-frequency bands (delta/theta) important?
    low_freq_imp = subband_importance_norm[:2].sum()  # cA4 + cD4
    print(f"  Delta+Theta (cA4+cD4): {low_freq_imp:.1%}  "
          f"{'✅ HIGH — consistent with slow-wave deception EEG literature' if low_freq_imp > 0.30 else '⚠️  LOW — high-frequency features dominate'}")
    
    # ── Most important single cell ─────────────────────────────
    best_ch_idx, best_band_idx = np.unravel_index(avg_all.argmax(), avg_all.shape)
    print(f"\n  🔝 Most important feature: {CHANNELS[best_ch_idx]} channel, {sub_band_labels[best_band_idx]} sub-band")
    
    print("\n" + "=" * 60)
    print("CONCLUSION for thesis:")
    print("If T7/T8 and Pz dominate, and low-freq bands are important,")
    print("the model has learned neurophysiologically plausible features.")
    print("If frontal channels (AF3/AF4) dominate, consider muscle artifacts.")
    print("=" * 60)

---
## Step 15: Save Results

Saves the importance matrices as CSV files for use in your thesis / further analysis.

In [ ]:
# ── Save importance matrices as CSV ───────────────────────
if lie_avg is not None:
    df_lie = pd.DataFrame(lie_avg, index=CHANNELS, columns=sub_band_labels)
    df_lie.to_csv('gradcam_lie_importance.csv')
    print("✓ Saved: gradcam_lie_importance.csv")

if truth_avg is not None:
    df_truth = pd.DataFrame(truth_avg, index=CHANNELS, columns=sub_band_labels)
    df_truth.to_csv('gradcam_truth_importance.csv')
    print("✓ Saved: gradcam_truth_importance.csv")

# ── Summary of all output files ───────────────────────────
print("\n📁 Files saved in Master_Thesis/ folder:")
print("  best_model.keras              — trained CNN weights")
print("  confusion_matrix.png          — test set confusion matrix")
print("  training_curves.png           — accuracy/loss over epochs")
print("  gradcam_heatmap.png           — main Grad-CAM visualisation")
print("  channel_importance.png        — per-channel bar chart")
print("  gradcam_lie_importance.csv    — importance matrix for Lie class")
print("  gradcam_truth_importance.csv  — importance matrix for Truth class")